In [30]:
import os
import collections
import numpy as np
from numpy.lib.format import open_memmap
from pathlib import Path
from tqdm import tqdm
import openwakeword
import openwakeword.data
import openwakeword.utils
import openwakeword.metrics
from openwakeword.utils import download_models
import scipy
import datasets
import matplotlib.pyplot as plt
import torch
from torch import nn
import IPython.display as ipd

In [28]:
cv_11 = datasets.load_dataset("mozilla-foundation/common_voice_11_0", "en", split="test", streaming=True)
cv_11 = cv_11.cast_column("audio", datasets.Audio(sampling_rate=16000, mono=True)) # convert to 16-khz
cv_11 = cv_11.cast_column("text", datasets.Value("string"))

# Convert and save clips (only first 5000)
limit = 5000
for i, example in tqdm(enumerate(cv_11), total=limit):
    if i >= limit:  # Stop if we've reached the limit
        break
    
    output = os.path.join("cv11_test_clips", example["path"][0:-4] + ".wav")
    os.makedirs(os.path.dirname(output), exist_ok=True)

    # Convert the audio array to 16-bit PCM format
    wav_data = (example["audio"]["array"] * 32767).astype(np.int16)
    
    # Save the audio clip as a .wav file
    scipy.io.wavfile.write(output, 16000, wav_data)

Reading metadata...: 16354it [00:00, 16400.60it/s]
100%|██████████| 5000/5000 [01:39<00:00, 50.16it/s]


In [33]:
download_models(
    target_directory="../models",
)

embedding_model.tflite: 100%|██████████| 1.33M/1.33M [00:00<00:00, 5.93MiB/s]
embedding_model.onnx: 100%|██████████| 1.33M/1.33M [00:00<00:00, 7.81MiB/s]
melspectrogram.tflite: 100%|██████████| 1.09M/1.09M [00:00<00:00, 8.66MiB/s]
melspectrogram.onnx: 100%|██████████| 1.09M/1.09M [00:00<00:00, 4.10MiB/s]
silero_vad.onnx: 100%|██████████| 1.81M/1.81M [00:00<00:00, 4.50MiB/s]
alexa_v0.1.tflite: 100%|██████████| 855k/855k [00:00<00:00, 8.32MiB/s]
alexa_v0.1.onnx: 100%|██████████| 854k/854k [00:00<00:00, 6.24MiB/s]
hey_mycroft_v0.1.tflite: 100%|██████████| 860k/860k [00:00<00:00, 5.13MiB/s]
hey_mycroft_v0.1.onnx: 100%|██████████| 858k/858k [00:00<00:00, 4.40MiB/s]
hey_jarvis_v0.1.tflite: 100%|██████████| 1.28M/1.28M [00:00<00:00, 7.56MiB/s]
hey_jarvis_v0.1.onnx: 100%|██████████| 1.27M/1.27M [00:00<00:00, 7.72MiB/s]
hey_rhasspy_v0.1.tflite: 100%|██████████| 416k/416k [00:00<00:00, 5.92MiB/s]
hey_rhasspy_v0.1.onnx: 100%|██████████| 204k/204k [00:00<00:00, 7.84MiB/s]
timer_v0.1.tflite: 100%|█

In [47]:
import os
#list current directory
os.listdir()
os.getcwd()
F = openwakeword.utils.AudioFeatures(melspec_model_path="/home/sebastian/Mine/dl/models/melspectrogram.onnx", embedding_model_path="/home/sebastian/Mine/dl/models/embedding_model.onnx")

In [48]:
negative_clips, negative_durations = openwakeword.data.filter_audio_paths(
    [
        "/home/sebastian/Mine/dl/data/fma_sample",
        "/home/sebastian/Mine/dl/data/fsd50k_sample",
        "/home/sebastian/Mine/dl/data/cv11_test_clips"
    ],
    min_length_secs = 1.0, # minimum clip length in seconds
    max_length_secs = 60*30, # maximum clip length in seconds
    duration_method = "header" # use the file header to calculate duration
)

print(f"{len(negative_clips)} negative clips after filtering, representing ~{sum(negative_durations)//3600} hours")

200it [00:00, 107339.83it/s]
100%|██████████| 200/200 [00:00<00:00, 2592.77it/s]
1000it [00:00, 127579.51it/s]
100%|██████████| 1000/1000 [00:00<00:00, 2929.62it/s]
1it [00:00, 2077.42it/s]
100%|██████████| 1/1 [00:00<00:00, 1235.80it/s]

1097 negative clips after filtering, representing ~4.0 hours


In [50]:
audio_dataset = datasets.Dataset.from_dict({"audio": negative_clips})
audio_dataset = audio_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))

In [52]:
# Get audio embeddings (features) for negative clips and save to .npy file
# Process files by batch and save to Numpy memory mapped file so that
# an array larger than the available system memory can be created

batch_size = 64 # number of files to load, compute features, and write to mmap at a time
clip_size = 3  # the desired window size (in seconds) for the trained openWakeWord model
N_total = int(sum(negative_durations)//clip_size) # maximum number of rows in mmap file
n_feature_cols = F.get_embedding_shape(clip_size)

output_file = "negative_features.npy"
output_array_shape = (N_total, n_feature_cols[0], n_feature_cols[1])
fp = open_memmap(output_file, mode='w+', dtype=np.float32, shape=output_array_shape)

row_counter = 0
for i in tqdm(np.arange(0, audio_dataset.num_rows, batch_size)):
    # Load data in batches and shape into rectangular array
    wav_data = [(j["array"]*32767).astype(np.int16) for j in audio_dataset[i:i+batch_size]["audio"]]
    wav_data = openwakeword.data.stack_clips(wav_data, clip_size=16000*clip_size).astype(np.int16)
    
    # Compute features (increase ncpu argument for faster processing)
    features = F.embed_clips(x=wav_data, batch_size=1024, ncpu=8)
    
    # Save computed features to mmap array file (stopping once the desired size is reached)
    if row_counter + features.shape[0] > N_total:
        fp[row_counter:min(row_counter+features.shape[0], N_total), :, :] = features[0:N_total - row_counter, :, :]
        fp.flush()
        break
    else:
        fp[row_counter:row_counter+features.shape[0], :, :] = features
        row_counter += features.shape[0]
        fp.flush()
        
# Trip empty rows from the mmapped array
openwakeword.data.trim_mmap(output_file)

 94%|█████████▍| 17/18 [00:33<00:01,  1.94s/it]
Trimming empty rows: 5it [00:00, 32.93it/s]                       


In [74]:
# Get positive example paths, filtering out clips that are too long or too short

positive_clips, durations = openwakeword.data.filter_audio_paths(
    [
        "/home/sebastian/Mine/dl/data/turn_on_the_office_lights"
    ],
    min_length_secs = 1.0, # minimum clip length in seconds
    max_length_secs = 2.0, # maximum clip length in seconds
    duration_method = "header" # use the file header to calculate duration
)

print(f"{len(positive_clips)} positive clips after filtering")

3388it [00:00, 233342.12it/s]
100%|██████████| 3388/3388 [00:00<00:00, 15401.63it/s]

3203 positive clips after filtering


In [95]:
# Define starting point for each positive clip based on its length, so that each one ends 
# between 0-200 ms from the end of the total window size chosen for the model.
# This results in the model being most confident in the prediction right after the
# end of the wakeword in the audio stream, reducing latency in operation.

# Get start and end positions for the positive audio in the full window
sr = 16000
total_length_seconds = 3 # must be the some window length as that used for the negative examples
total_length = int(sr*total_length_seconds)

jitters = (np.random.uniform(0, 0.2, len(positive_clips))*sr).astype(np.int32)
starts = [total_length - (int(np.ceil(i*sr))+j) for i,j in zip(durations, jitters)]
ends = [int(i*sr) + j for i, j in zip(durations, starts)]

# Create generator to mix the positive audio with background audio
batch_size = 8
mixing_generator = openwakeword.data.mix_clips_batch(
    foreground_clips = positive_clips,
    background_clips = negative_clips,
    combined_size = total_length,
    batch_size = batch_size,
    snr_low = 5,
    snr_high = 15,
    start_index = starts,
    volume_augmentation=True, # randomly scale the volume of the audio after mixing
)
print(f"Number of positive clips: {len(positive_clips)}")
print(f"Number of negative clips: {len(negative_clips)}")
#print torch version
print(torch.__version__)
try:
    mixed_clips, labels, background_clips = next(mixing_generator)
    print(f"Mixed Clips: {mixed_clips}")
    print(f"Labels: {labels}")
    print(f"Background Clips: {background_clips}")
except StopIteration:
    print("Generator has no more data.")



Number of positive clips: 3203
Number of negative clips: 1097
2.5.1+cu124


TypeError: _amax() got an unexpected keyword argument 'dim'